## Instrucciones de uso

## Introducción
Para este caso vamos a crear un Knowledge Base en AWS Bedrock que usara un S3 vectors para guardar la información, esto nos permitirá crear un RAG que consultará la información de acuerdo a lo que se le pregunte y un modelo de AI gestionará la respuesta.

## Servicios a usar
* S3
* S3 vectors
* Bedrock
* Amazon titan embed text v2

## Requisitos
Tener instalado AWS CLI configurado
Tener permisos IAM para realizar pasos

**nota:** para este caso usaremos la region **us-east-2**

#### Crear Knowledge Base con S3 Vectors

#### 1. crear Bucket S3 para guardar archivos fuente conocimiento
ingresar a la consola AWS y en el servicio S3 crear bucket y colocar archivos, en este caso el bucket se llama **kb-docs**

#### 2. crear S3 vector
ingresar a la consola AWS y en el servicio S3 crear vector bucket S3, en este caso se llama **mykb-vector-store-2**, para este ejemplo usamos estas opciones
* dimension 1024 si es titan v2 2:0
* metric distance cosine
* metadata filtering default

#### 3. tener archivo kb-config-v2.json

``` json
{
  "type": "VECTOR",
  "vectorKnowledgeBaseConfiguration": {
    "embeddingModelArn": "arn:aws:bedrock:us-east-2::foundation-model/amazon.titan-embed-text-v2:0"
  }
}
```
#### 4. tener archivo s3-config.json

``` json
{
    "type": "S3_VECTORS",
    "s3VectorsConfiguration": {
    "vectorBucketArn": "arn:aws:s3vectors:us-east-2:<id_account>:bucket/mykb-vector-store-2",
    "indexName": "mi-index"
}
```

#### 5. crear KB en bedrock y obtener el id (guardar id)
``` cmd
aws bedrock-agent create-knowledge-base --region us-east-2 --name "mi-kb-s3vectors-v2" --role-arn "arn:aws:iam::<id_account>:role/BedrockKBRoleS3Vectors" --knowledge-base-configuration file://kb-config-v2.json --storage-configuration file://s3-config.json
``` 

#### ver status de creacion
``` cmd
aws bedrock-agent get-knowledge-base --region us-east-2 --knowledge-base-id <id_kb>
```

#### 6. tener archivo ds-config.json
``` json
{
  "knowledgeBaseId": "<id_kb>",
  "name": "mi-ds",
  "dataSourceConfiguration": {
    "type": "S3",
    "s3Configuration": {
      "bucketArn": "arn:aws:s3:::mykb-docs"
    }
  },
  "dataDeletionPolicy": "DELETE"
}
```

#### 7. crear data source (guardar id)
``` cmd
aws bedrock-agent create-data-source --region us-east-2 --cli-input-json file://ds-config.json
```

#### 8. iniciar ingestion or sync (colocar los id de data source y KB, tomar id de ingestion)
``` cmd
aws bedrock-agent start-ingestion-job --region us-east-2 --knowledge-base-id <id_kb> --data-source-id <id_data_source>
```

#### 9. revisar como va ingestion
``` cmd
aws bedrock-agent get-ingestion-job --region us-east-2 --knowledge-base-id <id_kb> --data-source-id <id_data_source> --ingestion-job-id <id_ingestion>
```



#### Borrar Knowledge Bases y S3 vectors


#### 1. Tener el archivo ds-update.json
```json
{
  "name": "mi-ds",
  "dataSourceConfiguration": {
    "type": "S3",
    "s3Configuration": {
      "bucketArn": "arn:aws:s3:::mykb-docs"
    }
  },
  "dataDeletionPolicy": "RETAIN"
}
```

#### 2. ejecutar comando: (cambiar KB id y data source id)
``` cmd
aws bedrock-agent update-data-source --region us-east-2 --knowledge-base-id <id_kb> --data-source-id <id_data_source> --cli-input-json file://ds-update.json
```

#### 3. eliminar data source KB (cambiar KB id y data source id)
``` cmd
aws bedrock-agent delete-data-source --region us-east-2 --knowledge-base-id <id_kb> --data-source-id <id_data_source>
```

#### 4. verificar data source KB (cambiar KB id y data source id)
``` cmd
aws bedrock-agent get-data-source --region us-east-2 --knowledge-base-id <id_kb> --data-source-id <id_data_source>
```

#### 5. BORRAR KB
``` cmd
aws bedrock-agent delete-knowledge-base --region us-east-2 --knowledge-base-id <id_kb>
```

#### 6. REVISAR SI YA SE BORRO KB
``` cmd
aws bedrock-agent get-knowledge-base --region us-east-2 --knowledge-base-id <id_kb>
```

#### 7. borrar s3 vectors y el index del vector
Ir a la consola de AWS y manualmente borrar los buckets

#### 8. borrar s3 bucket general
Ir a la consola de AWS y manualmente borrar los buckets


